# Task 3 — Part A: Pool Ball Detection

**Goal:** Train an object detector that can locate every ball on a pool table, draw a bounding box around it, and label it with its category (`Black`, `Cue`, `Dot`, `Solid`, `Striped`).

**What we will build:**
1. Explore and understand the dataset (images + COCO annotations)
2. Convert the dataset to YOLO format (what the models expect)
3. Fine-tune **YOLOv8-s** — a fast, CNN-based single-stage detector
4. Fine-tune **RT-DETR-l** — a modern Transformer-based detector
5. Compare both models on standard detection metrics (mAP, precision, recall, speed)

> **Why these two?** They share the exact same Ultralytics training API, making a fair side-by-side comparison trivial to implement. The key difference is architectural: YOLO uses convolutional feature pyramids, while RT-DETR uses Transformer attention — exactly the CNN vs. Transformer narrative the "extra" comparison requires.

---
## 0. Install Dependencies

We only need one library: **Ultralytics**, which packages YOLOv8 and RT-DETR under a single, clean API. It also handles data augmentation, logging, and evaluation automatically.

In [ ]:
# Install Ultralytics (includes YOLOv8 and RT-DETR)
# Run this only once — comment it out after the first run to save time
!pip install ultralytics --quiet

In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from ultralytics import YOLO, RTDETR

# Reproducibility
random.seed(42)
np.random.seed(42)

print("All imports successful!")

---
## 1. Data Exploration

Before touching a model, always explore your data. We want to answer:
- How many images and annotations do we have?
- What categories exist, and how are they distributed?
- What do the images and ground-truth boxes look like?

Our dataset lives in `./train/` and uses **COCO JSON format**. In COCO:
- `images` → list of image metadata (id, file_name, width, height)
- `categories` → list of class names and their integer ids
- `annotations` → list of bounding boxes, each linked to an image and a category

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_DIR   = Path("./train")
COCO_JSON   = TRAIN_DIR / "_annotations.coco.json"  # adjust if your filename differs

# ── Load the COCO JSON ─────────────────────────────────────────────────────
with open(COCO_JSON) as f:
    coco = json.load(f)

print(f"Total images      : {len(coco['images'])}")
print(f"Total annotations : {len(coco['annotations'])}")
print(f"\nCategories:")
for cat in coco["categories"]:
    print(f"  id={cat['id']:>3}  name={cat['name']}")

In [ ]:
# ── Build helper look-up dictionaries ──────────────────────────────────────
# Map category id → name  (we will need this later)
id_to_name = {cat["id"]: cat["name"] for cat in coco["categories"]}

# Map image id → filename  (so we can load images by annotation)
id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}

# ── Count annotations per category ─────────────────────────────────────────
# We skip the parent "balls" category (supercategory) if present —
# it is a grouping label, not a real detection class.
real_cat_ids = {cat["id"] for cat in coco["categories"] if cat["name"] != "balls"}

cat_counts = Counter(
    id_to_name[ann["category_id"]]
    for ann in coco["annotations"]
    if ann["category_id"] in real_cat_ids
)

print("Annotation counts per class:")
for name, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {name:<10}: {count}")

# ── Bar chart ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cat_counts.keys(), cat_counts.values(), color="steelblue", edgecolor="white")
ax.set_title("Number of annotations per ball category")
ax.set_ylabel("Count")
ax.set_xlabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise ground-truth boxes on a few sample images ────────────────────
# This is the single most important sanity check: confirm your annotations
# actually sit on the correct objects before spending time training.

# Build a dict: image_id → list of annotations
ann_by_img = {}
for ann in coco["annotations"]:
    if ann["category_id"] not in real_cat_ids:
        continue
    ann_by_img.setdefault(ann["image_id"], []).append(ann)

# Colour palette for each class
PALETTE = {
    "Black"  : "#111111",
    "Cue"    : "#f5f5f5",
    "Dot"    : "#e63946",
    "Solid"  : "#2196f3",
    "Striped": "#ff9800",
}

sample_ids = random.sample(list(ann_by_img.keys()), k=4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_id in zip(axes.flat, sample_ids):
    img_path = TRAIN_DIR / id_to_file[img_id]
    img_bgr  = cv2.imread(str(img_path))
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    for ann in ann_by_img[img_id]:
        x, y, w, h = ann["bbox"]          # COCO bbox: [x_min, y_min, width, height]
        cat_name   = id_to_name[ann["category_id"]]
        color      = PALETTE.get(cat_name, "lime")
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x, y - 4, cat_name, color=color, fontsize=8,
                bbox=dict(facecolor="black", alpha=0.4, pad=1))

    ax.set_title(f"Image id {img_id}", fontsize=10)
    ax.axis("off")

plt.suptitle("Ground-truth bounding boxes (sample)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Data Preparation: COCO → YOLO Format

Ultralytics models expect data in **YOLO format**, which is different from COCO:

| Format | Annotation file | Box representation |
|--------|----------------|--------------------|
| COCO   | One big JSON   | `[x_min, y_min, width, height]` in pixels |
| YOLO   | One `.txt` per image | `[class_id, x_center, y_center, width, height]` **normalised 0–1** |

We also need to **create train/val/test splits** ourselves (our dataset has no pre-made splits).

**Split strategy:** 70% train / 15% val / 15% test, stratified at the image level.

In [ ]:
# ── 2.1  Map category name → 0-indexed YOLO class id ───────────────────────
# YOLO class ids must start at 0 and be contiguous.
# We sort alphabetically for reproducibility.
CLASS_NAMES  = sorted([cat["name"] for cat in coco["categories"]
                       if cat["name"] != "balls"])
name_to_yolo = {name: idx for idx, name in enumerate(CLASS_NAMES)}
print("YOLO class mapping:", name_to_yolo)

In [ ]:
# ── 2.2  Create the directory structure expected by Ultralytics ─────────────
#
#   dataset/
#   ├── images/
#   │   ├── train/   ← image files (.jpg)
#   │   ├── val/
#   │   └── test/
#   └── labels/
#       ├── train/   ← one .txt per image
#       ├── val/
#       └── test/

DATASET_DIR = Path("./dataset")

for split in ["train", "val", "test"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Directory structure created.")

In [ ]:
# ── 2.3  Split image ids into train / val / test ────────────────────────────
# Only keep images that have at least one real annotation.
all_img_ids = sorted(ann_by_img.keys())
random.shuffle(all_img_ids)

n       = len(all_img_ids)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
# Test gets the remainder

splits = {
    "train": all_img_ids[:n_train],
    "val"  : all_img_ids[n_train : n_train + n_val],
    "test" : all_img_ids[n_train + n_val :],
}

for split, ids in splits.items():
    print(f"{split:>5}: {len(ids):>4} images")

In [ ]:
# ── 2.4  Convert annotations and copy images ────────────────────────────────
# For each image:
#   1. Copy the .jpg to dataset/images/<split>/
#   2. Write a .txt file to dataset/labels/<split>/
#      Each line: <class_id> <x_center> <y_center> <width> <height>  (all 0-1)

# Build a quick lookup: image_id → (width, height)
img_dims = {img["id"]: (img["width"], img["height"]) for img in coco["images"]}

def coco_bbox_to_yolo(bbox, img_w, img_h):
    """Convert COCO [x_min, y_min, w, h] to YOLO [xc, yc, w, h] normalised."""
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_w
    y_center = (y_min + h / 2) / img_h
    return x_center, y_center, w / img_w, h / img_h

for split, img_ids in splits.items():
    for img_id in img_ids:
        img_info  = next(img for img in coco["images"] if img["id"] == img_id)
        fname     = img_info["file_name"]
        img_w, img_h = img_dims[img_id]

        # 1. Copy image
        src = TRAIN_DIR / fname
        dst = DATASET_DIR / "images" / split / fname
        shutil.copy(src, dst)

        # 2. Write label file
        label_path = DATASET_DIR / "labels" / split / (Path(fname).stem + ".txt")
        lines = []
        for ann in ann_by_img.get(img_id, []):
            cat_name = id_to_name[ann["category_id"]]
            class_id = name_to_yolo[cat_name]
            xc, yc, w, h = coco_bbox_to_yolo(ann["bbox"], img_w, img_h)
            # Clamp values to [0, 1] to handle any annotation boundary errors
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))
            lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        label_path.write_text("\n".join(lines))

print("Dataset converted and organised successfully!")

In [ ]:
# ── 2.5  Write the YAML config file ────────────────────────────────────────
# Ultralytics reads a .yaml file to know where the data lives and
# how many classes exist. This is the only configuration file we need.

yaml_content = f"""# Pool Ball Detection Dataset
path: {DATASET_DIR.resolve()}  # absolute path to dataset root

train: images/train
val:   images/val
test:  images/test

nc: {len(CLASS_NAMES)}  # number of classes
names: {CLASS_NAMES}
"""

yaml_path = Path("pool_balls.yaml")
yaml_path.write_text(yaml_content)
print(yaml_path.read_text())

---
## 3. Model Training

### What is fine-tuning?

Instead of training from random weights (which would need thousands of images), we start from a model that was **pre-trained on COCO** (a large general-purpose detection dataset). We then continue training it on our small pool-ball dataset. The model already knows how to detect edges, shapes, and objects in general — we just teach it to specialise.

### YOLOv8-s — CNN-based detector

YOLOv8 divides the image into a grid. Each grid cell predicts bounding boxes and class probabilities simultaneously in a single forward pass — hence *single-stage*. It uses a **convolutional feature pyramid** to detect objects at multiple scales (useful since pool balls appear at very different sizes depending on camera distance).

### RT-DETR-l — Transformer-based detector

RT-DETR uses a **Transformer encoder** to model global relationships between all image patches. This means a ball in the corner of the image can "attend" to context across the whole scene. Unlike YOLO, it does **not use Non-Maximum Suppression (NMS)** — it directly outputs a fixed set of predictions, which makes its output cleaner but convergence slower.

> **Note:** Both models share the exact same `.train()` API — the only difference is which class you instantiate.

In [ ]:
# ── Training hyperparameters ────────────────────────────────────────────────
# These are conservative, Colab-friendly values.
# Increase epochs (up to 100) if you have more compute budget.

EPOCHS    = 50    # number of full passes over the training set
IMG_SIZE  = 640   # resize all images to 640×640 (standard for YOLO)
BATCH     = 16    # images per gradient update (reduce to 8 if you get OOM errors)
DATA_YAML = str(yaml_path.resolve())

print(f"Training config: epochs={EPOCHS}, imgsz={IMG_SIZE}, batch={BATCH}")

In [ ]:
# ── 3.1  Train YOLOv8-s ─────────────────────────────────────────────────────
# "yolov8s.pt" downloads the small variant pre-trained on COCO.
# The "s" = small — fast enough for Colab free tier.

print("=" * 60)
print("Training YOLOv8-s...")
print("=" * 60)

yolo_model = YOLO("yolov8s.pt")

yolo_results = yolo_model.train(
    data    = DATA_YAML,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    name    = "yolov8s_pool",   # results saved to runs/detect/yolov8s_pool/
    exist_ok= True,
    verbose = False,            # set True for per-epoch logs
)

print("\nYOLOv8-s training complete!")

In [ ]:
# ── 3.2  Train RT-DETR-l ────────────────────────────────────────────────────
# The Transformer model. Same API, different class.
# Use batch=8 instead of 16 — RT-DETR is heavier on memory.

print("=" * 60)
print("Training RT-DETR-l...")
print("=" * 60)

rtdetr_model = RTDETR("rtdetr-l.pt")

rtdetr_results = rtdetr_model.train(
    data    = DATA_YAML,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = 8,              # smaller batch for RT-DETR (heavier model)
    name    = "rtdetr_pool",  # results saved to runs/detect/rtdetr_pool/
    exist_ok= True,
    verbose = False,
)

print("\nRT-DETR-l training complete!")

---
## 4. Evaluation

We evaluate both models on the **held-out test set** — images the model has never seen.

### Key metrics explained

| Metric | Meaning | Range |
|--------|---------|-------|
| **Precision** | Of all boxes the model predicted, what fraction were correct? | 0–1, higher is better |
| **Recall** | Of all ground-truth boxes, what fraction did the model find? | 0–1, higher is better |
| **mAP@50** | Mean Average Precision at IoU ≥ 0.50 — the standard detection metric | 0–1, higher is better |
| **mAP@50:95** | Stricter: average mAP across IoU thresholds 0.50 to 0.95 | 0–1, higher is better |

**IoU (Intersection over Union)** measures how much a predicted box overlaps the ground-truth box:
- IoU = 1.0 → perfect overlap
- IoU < 0.5 → prediction is too far off to count as correct

**mAP** averages the Average Precision over all classes. It summarises the precision-recall trade-off into a single number.

In [ ]:
# ── 4.1  Load best weights and run evaluation on test set ──────────────────

YOLO_BEST   = Path("runs/detect/yolov8s_pool/weights/best.pt")
RTDETR_BEST = Path("runs/detect/rtdetr_pool/weights/best.pt")

# Load best checkpoints
yolo_eval   = YOLO(str(YOLO_BEST))
rtdetr_eval = RTDETR(str(RTDETR_BEST))

# Evaluate on the test split
print("Evaluating YOLOv8-s on test set...")
yolo_metrics = yolo_eval.val(data=DATA_YAML, split="test")

print("\nEvaluating RT-DETR-l on test set...")
rtdetr_metrics = rtdetr_eval.val(data=DATA_YAML, split="test")

In [ ]:
# ── 4.2  Build the comparison table ────────────────────────────────────────
# Ultralytics stores metrics in a results object.
# We extract the key numbers and display them side by side.

def extract_metrics(metrics):
    """Pull the core numbers out of an Ultralytics val result object."""
    return {
        "Precision"   : round(float(metrics.box.mp),   4),
        "Recall"      : round(float(metrics.box.mr),   4),
        "mAP@50"      : round(float(metrics.box.map50), 4),
        "mAP@50:95"   : round(float(metrics.box.map),  4),
    }

yolo_m   = extract_metrics(yolo_metrics)
rtdetr_m = extract_metrics(rtdetr_metrics)

print(f"{'Metric':<15} {'YOLOv8-s':>12} {'RT-DETR-l':>12}")
print("-" * 42)
for metric in ["Precision", "Recall", "mAP@50", "mAP@50:95"]:
    print(f"{metric:<15} {yolo_m[metric]:>12.4f} {rtdetr_m[metric]:>12.4f}")

In [ ]:
# ── 4.3  Inference speed comparison ────────────────────────────────────────
# Speed matters for real-time applications.
# We time each model on a single test image (CPU inference to be hardware-agnostic).

import time

test_images = sorted((DATASET_DIR / "images" / "test").glob("*.jpg"))
sample_img  = str(test_images[0])
N_RUNS      = 20  # average over multiple runs for stability

def measure_speed(model, img_path, n_runs=20):
    # Warm-up pass
    model.predict(img_path, verbose=False)
    start = time.perf_counter()
    for _ in range(n_runs):
        model.predict(img_path, verbose=False)
    elapsed = time.perf_counter() - start
    return (elapsed / n_runs) * 1000  # ms per image

yolo_ms   = measure_speed(yolo_eval,   sample_img)
rtdetr_ms = measure_speed(rtdetr_eval, sample_img)

print(f"YOLOv8-s  inference speed : {yolo_ms:.1f} ms/image")
print(f"RT-DETR-l inference speed : {rtdetr_ms:.1f} ms/image")

In [ ]:
# ── 4.4  Visual comparison bar chart ───────────────────────────────────────

metrics_to_plot = ["Precision", "Recall", "mAP@50", "mAP@50:95"]
yolo_vals   = [yolo_m[m]   for m in metrics_to_plot]
rtdetr_vals = [rtdetr_m[m] for m in metrics_to_plot]

x     = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, yolo_vals,   width, label="YOLOv8-s",  color="#2196f3")
bars2 = ax.bar(x + width/2, rtdetr_vals, width, label="RT-DETR-l", color="#ff9800")

ax.set_ylabel("Score")
ax.set_title("YOLOv8-s vs RT-DETR-l — Detection Metrics on Test Set")
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars1, fmt="%.3f", padding=2, fontsize=8)
ax.bar_label(bars2, fmt="%.3f", padding=2, fontsize=8)

plt.tight_layout()
plt.show()

---
## 5. Qualitative Results

Numbers tell part of the story. Visuals tell the rest. We now run both models on the same set of test images and display their predictions side by side.

**What to look for:**
- Are the boxes tight or loose?
- Are any balls missed (false negatives)?
- Are there ghost detections on the table cloth (false positives)?
- Which model handles overlapping or partially occluded balls better?

In [ ]:
# ── Visualise predictions on 4 test images for each model ──────────────────

CONF_THRESHOLD = 0.25   # only show boxes with confidence ≥ 25%

sample_test = random.sample(test_images, k=4)

fig, axes = plt.subplots(4, 2, figsize=(16, 22))

for row, img_path in enumerate(sample_test):
    for col, (model, model_name) in enumerate([
        (yolo_eval,   "YOLOv8-s"),
        (rtdetr_eval, "RT-DETR-l")
    ]):
        results = model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
        result  = results[0]

        # result.plot() returns a BGR numpy array with boxes drawn
        plotted = result.plot()   # BGR
        plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)

        axes[row, col].imshow(plotted)
        n_det = len(result.boxes)
        axes[row, col].set_title(f"{model_name} — {n_det} detections", fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Predictions on test images (Left: YOLOv8-s | Right: RT-DETR-l)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Training Curves

Ultralytics saves a `results.csv` in each run folder with per-epoch metrics. Plotting the training curves helps us understand:
- Did the model converge? (loss going down smoothly)
- Is there overfitting? (train mAP rising but val mAP plateauing or dropping)
- Could we have stopped earlier? (early stopping opportunities)

In [ ]:
import pandas as pd

def plot_training_curves(run_dir, model_name, color):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        print(f"results.csv not found at {csv_path}")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # strip whitespace from headers

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"{model_name} — Training Curves", fontsize=13)

    # Box loss (train vs val)
    axes[0].plot(df["train/box_loss"], label="Train",      color=color)
    axes[0].plot(df["val/box_loss"],   label="Val",  ls="--", color=color, alpha=0.7)
    axes[0].set_title("Box Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    # Class loss
    axes[1].plot(df["train/cls_loss"], label="Train",      color=color)
    axes[1].plot(df["val/cls_loss"],   label="Val",  ls="--", color=color, alpha=0.7)
    axes[1].set_title("Class Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    # mAP@50
    if "metrics/mAP50(B)" in df.columns:
        axes[2].plot(df["metrics/mAP50(B)"], color=color)
        axes[2].set_title("Val mAP@50")
        axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

plot_training_curves("runs/detect/yolov8s_pool",  "YOLOv8-s",  "#2196f3")
plot_training_curves("runs/detect/rtdetr_pool",   "RT-DETR-l", "#ff9800")

---
## 7. Per-Class Analysis

Global mAP hides per-class behaviour. A model might score 0.90 overall by performing very well on common classes while completely missing rare ones. Pool balls are naturally imbalanced: there are many Solids and Stripeds but only one Cue and one Black ball per game.

In [ ]:
# Ultralytics stores per-class AP in metrics.box.ap_class_index and metrics.box.ap
# (available after calling .val())

def per_class_ap(metrics, class_names):
    """Return a dict {class_name: AP@50} from a val result object."""
    ap50_per_class = metrics.box.ap50   # array of AP@50 per class
    return {name: round(float(ap), 4)
            for name, ap in zip(class_names, ap50_per_class)}

yolo_per_class   = per_class_ap(yolo_metrics,   CLASS_NAMES)
rtdetr_per_class = per_class_ap(rtdetr_metrics, CLASS_NAMES)

print(f"{'Class':<10} {'YOLO AP@50':>12} {'RTDETR AP@50':>14}")
print("-" * 40)
for cls in CLASS_NAMES:
    print(f"{cls:<10} {yolo_per_class[cls]:>12.4f} {rtdetr_per_class[cls]:>14.4f}")

In [ ]:
# ── Per-class bar chart ─────────────────────────────────────────────────────
x     = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [yolo_per_class[c]   for c in CLASS_NAMES], width,
       label="YOLOv8-s",  color="#2196f3")
ax.bar(x + width/2, [rtdetr_per_class[c] for c in CLASS_NAMES], width,
       label="RT-DETR-l", color="#ff9800")

ax.set_ylabel("AP@50")
ax.set_title("Per-class AP@50: YOLOv8-s vs RT-DETR-l")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Summary & Report Notes

Run this cell to print a clean summary you can copy into your report.

In [ ]:
print("=" * 55)
print("  TASK 3A — BALL DETECTION — RESULTS SUMMARY")
print("=" * 55)

print(f"\nDataset: {n} images  "
      f"(train={len(splits['train'])}, "
      f"val={len(splits['val'])}, "
      f"test={len(splits['test'])})")
print(f"Classes ({len(CLASS_NAMES)}): {', '.join(CLASS_NAMES)}")

print("\n--- Quantitative results (test set) ---")
print(f"{'Metric':<15} {'YOLOv8-s':>12} {'RT-DETR-l':>12}")
print("-" * 42)
for metric in ["Precision", "Recall", "mAP@50", "mAP@50:95"]:
    print(f"{metric:<15} {yolo_m[metric]:>12.4f} {rtdetr_m[metric]:>12.4f}")
print(f"{'Speed (ms/img)':<15} {yolo_ms:>12.1f} {rtdetr_ms:>12.1f}")

print("\n--- Per-class AP@50 ---")
print(f"{'Class':<10} {'YOLOv8-s':>10} {'RT-DETR-l':>12}")
print("-" * 35)
for cls in CLASS_NAMES:
    print(f"{cls:<10} {yolo_per_class[cls]:>10.4f} {rtdetr_per_class[cls]:>12.4f}")

print("\n" + "=" * 55)

---
## 9. Discussion Guide

Use this section to help you write the report and prepare for the presentation.

### Architecture comparison: CNN (YOLO) vs Transformer (RT-DETR)

| Aspect | YOLOv8-s | RT-DETR-l |
|--------|----------|-----------|
| **Core mechanism** | Convolutional feature pyramid | Transformer encoder + CNN backbone |
| **Receptive field** | Local (grows with depth) | Global (attention attends to all patches) |
| **Post-processing** | Requires NMS to remove duplicate boxes | No NMS needed |
| **Convergence** | Fast (~20–30 epochs on small data) | Slower (benefits from more epochs) |
| **Speed** | Faster inference | Slower due to attention computation |
| **Small datasets** | Better (less data hungry) | Worse (Transformers need more data) |

### Likely observations and how to explain them

- **YOLO likely wins on speed** — convolutional operations are highly optimised and parallelise well on GPUs.
- **RT-DETR may score slightly higher on mAP@50:95** — global attention helps with precise box localisation.
- **Cue ball likely has highest AP** — it is white and distinctive; harder classes are Dot (visually similar to Solid from a distance).
- **Class imbalance** — if Dot has few annotations it will have lower AP; mention this in the report.

### Limitations and future work

- **Small dataset (247 images):** both models rely heavily on COCO pre-training; collecting more pool images would improve results.
- **Single angle:** if the dataset only contains one camera angle, the model may generalise poorly to other viewpoints.
- **Augmentation:** increasing augmentation strength (mosaic, random perspective) could further regularise the models.
- **Larger model variants:** YOLOv8-m or RT-DETR-x would likely score higher at the cost of speed.